# Notebook 4b: Build Matched Subset (shared by CLIP and VLM)

# Load Config and Case-Split Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
from pathlib import Path

import pandas as pd

CONFIG = Path("/content/drive/MyDrive/Surgical-VLM/configs/config.json")

with open(CONFIG) as f:
    config = json.load(f)

PROCESSED_DIR = Path(config["processed_dir"])

# These already have a leakage-free, case-level train/val/test split
# (built in 4_OpenCLIPDataset.ipynb) and the final 8-task list
# (after removing incompatible tasks in 3_MasterDataset.ipynb)
train_df = pd.read_parquet(PROCESSED_DIR / "openclip_train.parquet")
val_df = pd.read_parquet(PROCESSED_DIR / "openclip_val.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "openclip_test.parquet")

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

print("\nTrain task distribution:")
print(train_df["task"].value_counts())


Train: (505527, 17)
Val  : (50986, 17)
Test : (105449, 17)

Train task distribution:
task
Triplet Recognition             116651
Instrument Recognition           87559
Safety Assessment                85878
Action Recognition               76830
Phase Recognition                60801
Surgical Image Captioning        53962
Tissue and Organ Recognition     23846
Name: count, dtype: int64


# Extract and Validate Closed-Vocabulary Label Sets

Parses `gt_label` into canonical label sets per task (handles both the plain-string and dict-wrapped formats seen in the data), then sanity-checks the extracted vocabulary sizes against CholecT50's official, published taxonomy (6 instruments, 10 verbs, 15 targets, 7 phases) — confirming the parser is pulling real categorical values rather than noise, before this feeds into evaluation.

In [ ]:
import json
import ast

def _parse_gt_label(raw):

    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None
    if isinstance(raw, list):
        return raw
    try:
        return json.loads(raw)
    except (ValueError, TypeError):
        try:
            return ast.literal_eval(raw)
        except (ValueError, SyntaxError):
            return None

def extract_label_set(task, gt_label_raw):
    """Returns a frozenset of canonical label strings, or None if unavailable
    (e.g. Surgical Image Captioning, which has no structured gt_label)."""
    parsed = _parse_gt_label(gt_label_raw)
    if parsed is None or len(parsed) == 0:
        return None

    item = parsed[0]

    if task == "Phase Recognition":
        if isinstance(item, dict):
            return frozenset([item.get("phase")])
        return frozenset([item])

    if task == "Tissue and Organ Recognition":
        if isinstance(item, dict):
            return frozenset(item.get("organs", item.get("organ", [])))
        return frozenset([item])

    if task == "Instrument Recognition":
        if isinstance(item, dict):
            return frozenset(item.get("instruments", []))
        return None

    if task == "Action Recognition":
        if isinstance(item, dict):
            return frozenset(item.get("actions", []))
        return None

    if task == "Safety Assessment":
        if isinstance(item, dict):
            cvs = item.get("critical_view_safety", {})
            return frozenset(cvs.items())
        return None

    if task == "Triplet Recognition":
        if isinstance(item, dict) and "triplets" in item:
            return frozenset(
                "{}_{}_{}".format(t["instrument"], t["verb"], t["target"])
                for t in item["triplets"]
            )
        elif isinstance(item, str):
            out = set()
            for p in item.split(","):
                toks = p.strip().split(" ", 2)
                if len(toks) == 3:
                    out.add(f"{toks[0]}_{toks[1]}_{toks[2]}")
            return frozenset(out)
        return None

    return None  # Surgical Image Captioning: no structured label


for df_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df["label_set"] = df.apply(
        lambda r: extract_label_set(r["task"], r["gt_label"]), axis=1
    )

print("Vocabulary sizes found in TRAIN (sanity check against official CholecT50 counts):\n")
for task in train_df["task"].unique():
    task_labels = train_df[train_df.task == task]["label_set"].dropna()
    if len(task_labels) == 0:
        print(f"{task:35s} -> no structured gt_label (expected for Captioning)")
        continue
    vocab = set()
    for s in task_labels:
        vocab |= set(s)
    print(f"{task:35s} -> {len(vocab)} unique values: {sorted(str(v) for v in vocab)[:8]}{' ...' if len(vocab) > 8 else ''}")


Vocabulary sizes found in TRAIN (sanity check against official CholecT50 counts):

Action Recognition                  -> 10 unique values: ['aspirate', 'clip', 'coagulate', 'cut', 'dissect', 'grasp', 'irrigate', 'null verb'] ...
Safety Assessment                   -> 6 unique values: ["('criterion_1', 'No')", "('criterion_1', 'Yes')", "('criterion_2', 'No')", "('criterion_2', 'Yes')", "('criterion_3', 'No')", "('criterion_3', 'Yes')"]
Phase Recognition                   -> 8 unique values: ['Calot Triangle Dissection', 'Cleaning Coagulation', 'Clipping Cutting', 'Gallbladder Dissection', 'Gallbladder Extraction', 'Gallbladder Packaging', 'Gallbladder Retraction', 'Preparation']
Tissue and Organ Recognition        -> 31 unique values: ['abdominal wall cavity', 'adhesion', 'cystic artery', 'cystic duct', 'cystic duct, fluid', 'cystic plate', 'fluid', 'gallbladder'] ...
Instrument Recognition              -> 7 unique values: ['bipolar', 'clipper', 'grasper', 'hook', 'irrigator', 'scissor

In [ ]:
EXCLUDED_TASKS = ["Instrument Localization"]

train_df = train_df[~train_df["task"].isin(EXCLUDED_TASKS)].reset_index(drop=True)
val_df = val_df[~val_df["task"].isin(EXCLUDED_TASKS)].reset_index(drop=True)
test_df = test_df[~test_df["task"].isin(EXCLUDED_TASKS)].reset_index(drop=True)

print("Final task list:")
print(train_df["task"].unique())


Final task list:
['Action Recognition' 'Safety Assessment' 'Phase Recognition'
 'Tissue and Organ Recognition' 'Instrument Recognition'
 'Triplet Recognition' 'Surgical Image Captioning']


In [ ]:
if measured_samples_per_hour and measured_cu_per_hour:
    affordable_hours = vlm_training_cu_budget / measured_cu_per_hour
    affordable_samples_total = affordable_hours * measured_samples_per_hour
    affordable_samples_per_epoch = affordable_samples_total / num_epochs_planned
    suggested_train_cap = int(affordable_samples_per_epoch / 8)  # 8 tasks

    print(f"Affordable training hours   : {affordable_hours:.1f}")
    print(f"Affordable samples (total)  : {affordable_samples_total:,.0f}")
    print(f"Affordable samples/epoch    : {affordable_samples_per_epoch:,.0f}")
    print(f"Suggested TRAIN_CAP per task: {suggested_train_cap:,}")
    print("\n Set TRAIN_CAP to this value in the cell below before capping.")
else:
    print("Run the pilot in 8_FineTuneVisionLLM.ipynb first, then fill in the "
          "two measured_* variables above before trusting TRAIN_CAP below.")


Affordable training hours   : 36.5
Affordable samples (total)  : 104,599
Affordable samples/epoch    : 104,599
Suggested TRAIN_CAP per task: 13,074

→ Set TRAIN_CAP to this value in the cell below before capping.


In [ ]:
TRAIN_CAP = 13000
VAL_CAP = 1500
TEST_CAP = 800

def soft_cap_per_task(df, cap, seed=42):
    return (
        df.groupby("task", group_keys=False)
        .apply(lambda g: g if len(g) <= cap else g.sample(n=cap, random_state=seed))
        .reset_index(drop=True)
    )

matched_train = soft_cap_per_task(train_df, TRAIN_CAP)
matched_val = soft_cap_per_task(val_df, VAL_CAP)
matched_test = soft_cap_per_task(test_df, TEST_CAP)

print("Matched train:", matched_train.shape, "(was", train_df.shape[0], ")")
print("Matched val  :", matched_val.shape, "(was", val_df.shape[0], ")")
print("Matched test :", matched_test.shape, "(was", test_df.shape[0], ")")


/tmp/ipykernel_6294/2442079361.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g if len(g) <= cap else g.sample(n=cap, random_state=seed))
/tmp/ipykernel_6294/2442079361.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g if len(g) <= cap else g.sample(n=cap, random_state=seed))


Matched train: (91000, 18) (was 505527 )
Matched val  : (9000, 18) (was 50986 )
Matched test : (5600, 18) (was 105449 )


/tmp/ipykernel_6294/2442079361.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g if len(g) <= cap else g.sample(n=cap, random_state=seed))


# Verify Balance and Leakage

In [ ]:
print("Matched train per task:")
print(matched_train["task"].value_counts())

print("\nMax/min ratio:",
      matched_train["task"].value_counts().max() / matched_train["task"].value_counts().min())

# Re-verify no case leakage survived the capping (it shouldn't — capping only
# drops rows, it never moves a case between splits — but check anyway)
assert set(matched_train.case_id).isdisjoint(matched_val.case_id)
assert set(matched_train.case_id).isdisjoint(matched_test.case_id)
assert set(matched_val.case_id).isdisjoint(matched_test.case_id)

print("\n No case leakage in matched splits.")


Matched train per task:
task
Action Recognition              13000
Instrument Recognition          13000
Phase Recognition               13000
Safety Assessment               13000
Surgical Image Captioning       13000
Tissue and Organ Recognition    13000
Triplet Recognition             13000
Name: count, dtype: int64

Max/min ratio: 1.0

✓ No case leakage in matched splits.


# Save Matched Subset

Both `6_FineTuneOpenCLIP.ipynb` and `8_FineTuneVisionLLM.ipynb` should load `matched_train.parquet` / `matched_val.parquet` instead of building their own subsets. Both `7_OpenCLIP_evaluation.ipynb` and the eval half of `8_FineTuneVisionLLM.ipynb` should load `matched_test.parquet`.

In [ ]:
matched_train["label_set"] = matched_train["label_set"].apply(lambda x: list(x) if isinstance(x, frozenset) else [])
matched_val["label_set"] = matched_val["label_set"].apply(lambda x: list(x) if isinstance(x, frozenset) else [])
matched_test["label_set"] = matched_test["label_set"].apply(lambda x: list(x) if isinstance(x, frozenset) else [])

matched_train.to_parquet(PROCESSED_DIR / "matched_train.parquet", index=False)
matched_val.to_parquet(PROCESSED_DIR / "matched_val.parquet", index=False)
matched_test.to_parquet(PROCESSED_DIR / "matched_test.parquet", index=False)

print("Saved:")
print(PROCESSED_DIR / "matched_train.parquet")
print(PROCESSED_DIR / "matched_val.parquet")
print(PROCESSED_DIR / "matched_test.parquet")

Saved:
/content/drive/MyDrive/Surgical-VLM/processed/matched_train.parquet
/content/drive/MyDrive/Surgical-VLM/processed/matched_val.parquet
/content/drive/MyDrive/Surgical-VLM/processed/matched_test.parquet
